In [1]:
import gurobipy as gp
from gurobipy import GRB
import math
import numpy as np

def solve_cvrp_gurobi(instance_name, nodes, N_v, C):
    """
    Solves the CVRP using Gurobi.
    nodes: list of (x,y) tuples. Index 0 must be the depot.
    """
    n = len(nodes)
    
    # 1. Calculate Euclidean Distance Matrix
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i,j] = math.sqrt((nodes[i][0] - nodes[j][0])**2 + (nodes[i][1] - nodes[j][1])**2)
            
    # 2. Initialize Gurobi Model
    m = gp.Model(instance_name)
    m.setParam('OutputFlag', 0) # Suppresses the long Gurobi log output
    
    # 3. Decision Variables
    # x[i,j] = 1 if a vehicle travels directly from node i to node j
    x = m.addVars(n, n, vtype=GRB.BINARY, name="x")
    
    # u[i] = continuous variable to track the "load" (customers visited) of a vehicle at node i
    u = m.addVars(n, vtype=GRB.CONTINUOUS, lb=1, ub=C, name="u")
    
    # 4. Objective: Minimize total distance
    m.setObjective(gp.quicksum(dist[i,j] * x[i,j] for i in range(n) for j in range(n)), GRB.MINIMIZE)
    
    # 5. Constraints
    m.addConstrs((x[i,i] == 0 for i in range(n)), "no_self_loops") # Cannot travel to itself
    
    # Every customer is entered exactly once and exited exactly once
    m.addConstrs((gp.quicksum(x[i,j] for i in range(n)) == 1 for j in range(1, n)), "enter_once")
    m.addConstrs((gp.quicksum(x[i,j] for j in range(n)) == 1 for i in range(1, n)), "exit_once")
    
    # Depot constraints: Use AT MOST N_v vehicles
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) <= N_v, "depot_out")
    m.addConstr(gp.quicksum(x[i,0] for i in range(1, n)) <= N_v, "depot_in")
    
    # Flow balance at depot (vehicles leaving must equal vehicles returning)
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) == gp.quicksum(x[i,0] for i in range(1, n)), "depot_bal")
    
    # MTZ Sub-tour Elimination and Capacity Constraint
    # This prevents disconnected loops and enforces that no route visits more than C customers
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                m.addConstr(u[i] - u[j] + C * x[i,j] <= C - 1, f"mtz_{i}_{j}")
                
    # 6. Solve
    m.optimize()
    
    # 7. Extract and Format Results
    if m.Status == GRB.OPTIMAL:
        print(f"--- Optimal Solution for {instance_name} ---")
        print(f"Total Distance: {m.ObjVal:.2f}")
        
        # Find active edges
        active_edges = [(i, j) for i in range(n) for j in range(n) if x[i,j].x > 0.5]
        
        # Trace routes starting from depot
        starts = [j for (i, j) in active_edges if i == 0]
        for route_idx, start_node in enumerate(starts):
            route = [0, start_node]
            current = start_node
            while True:
                next_node = [j for (i, j) in active_edges if i == current][0]
                route.append(next_node)
                current = next_node
                if current == 0: # Returned to depot
                    break
            
            # Format to match Hackathon required output
            route_str = ", ".join(map(str, route))
            print(f"r{route_idx + 1}: {route_str}")
    else:
        print(f"No optimal solution found for {instance_name}.")

In [2]:
# Instance 1: N_v = 2, C = 5
nodes_inst1 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3)    # Node 3
]

solve_cvrp_gurobi("Instance 1", nodes_inst1, N_v=2, C=5)

Restricted license - for non-production use only - expires 2027-11-29
--- Optimal Solution for Instance 1 ---
Total Distance: 21.74
r1: 0, 3, 2, 1, 0


In [3]:
# Instance 2: N_v = 2, C = 2
nodes_inst2 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3)    # Node 3
]

solve_cvrp_gurobi("Instance 2", nodes_inst2, N_v=2, C=2)

--- Optimal Solution for Instance 2 ---
Total Distance: 26.18
r1: 0, 2, 1, 0
r2: 0, 3, 0


In [4]:
# Instance 3: N_v = 3, C = 2
nodes_inst3 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (2, 3),   # Node 3
    (5, 7),   # Node 4
    (2, 4),   # Node 5
    (2, -3)   # Node 6
]

solve_cvrp_gurobi("Instance 3", nodes_inst3, N_v=3, C=2)

--- Optimal Solution for Instance 3 ---
Total Distance: 49.50
r1: 0, 1, 2, 0
r2: 0, 3, 6, 0
r3: 0, 4, 5, 0


In [5]:
# Instance 4: N_v = 4, C = 3
nodes_inst4 = [
    (0, 0),   # Node 0: Depot
    (-2, 2),  # Node 1
    (-5, 8),  # Node 2
    (6, 3),   # Node 3
    (4, 4),   # Node 4
    (3, 2),   # Node 5
    (0, 2),   # Node 6
    (-2, 3),  # Node 7
    (-4, 3),  # Node 8
    (2, 3),   # Node 9
    (2, 7),   # Node 10
    (-2, 5),  # Node 11
    (-1, 4)   # Node 12
]

solve_cvrp_gurobi("Instance 4", nodes_inst4, N_v=4, C=3)

--- Optimal Solution for Instance 4 ---
Total Distance: 58.18
r1: 0, 3, 4, 10, 0
r2: 0, 5, 9, 6, 0
r3: 0, 11, 2, 8, 0
r4: 0, 12, 7, 1, 0
